# Daily Challenge – Week 5, Day 5: Building Your First Neural Network on the MNIST Dataset
Runs best in **Google Colab** (TensorFlow is pre-installed). Training takes about 1–2 minutes on a CPU.

## 1. Load and Preprocess the MNIST Dataset

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.datasets import mnist
from tensorflow.keras.utils import to_categorical
from sklearn.metrics import confusion_matrix, classification_report

tf.random.set_seed(42)
np.random.seed(42)

# MNIST already comes split into a training set (60,000 images) and a test set (10,000 images)
(x_train, y_train), (x_test, y_test) = mnist.load_data()
print("Training images:", x_train.shape, "| Test images:", x_test.shape)

# Normalize pixel values from 0–255 to 0–1
x_train = x_train.astype("float32") / 255.0
x_test = x_test.astype("float32") / 255.0

# One-hot encode the labels
y_train_cat = to_categorical(y_train, 10)
y_test_cat = to_categorical(y_test, 10)
print("Label 5 as one-hot:", y_train_cat[0])

In [ ]:
# Sample images with their labels
plt.figure(figsize=(10, 4))
for i in range(10):
    plt.subplot(2, 5, i + 1)
    plt.imshow(x_train[i], cmap="gray")
    plt.title(f"Label: {y_train[i]}")
    plt.axis("off")
plt.tight_layout()
plt.show()

## 2. Build a Fully Connected Neural Network

In [ ]:
def build_model(hidden1=128, hidden2=64, learning_rate=0.001):
    model = models.Sequential([
        layers.Input(shape=(28, 28)),
        layers.Flatten(),                             # 28x28 → 784
        layers.Dense(hidden1, activation="relu"),     # hidden layer 1
        layers.Dense(hidden2, activation="relu"),     # hidden layer 2
        layers.Dense(10, activation="softmax"),       # one probability per digit
    ])
    model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=learning_rate),
                  loss="categorical_crossentropy",
                  metrics=["accuracy"])
    return model

model = build_model()
model.summary()

## 3. Train the Neural Network
10 epochs, with 10% of the training data held out as a **validation set** to track performance during training.

In [ ]:
history = model.fit(x_train, y_train_cat, epochs=10, batch_size=64, validation_split=0.1, verbose=2)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(history.history["loss"], label="training")
axes[0].plot(history.history["val_loss"], label="validation")
axes[0].set_title("Loss per epoch")
axes[0].set_xlabel("Epoch")
axes[0].legend()

axes[1].plot(history.history["accuracy"], label="training")
axes[1].plot(history.history["val_accuracy"], label="validation")
axes[1].set_title("Accuracy per epoch")
axes[1].set_xlabel("Epoch")
axes[1].legend()
plt.show()

**What to look for:** the training loss keeps falling every epoch. The validation loss falls quickly at first and then flattens (or starts rising slightly) after a few epochs – a sign that the model begins to **overfit**: it keeps getting better on the training images, but no longer improves on new ones.

## 4. Evaluate the Model's Performance

In [ ]:
test_loss, test_acc = model.evaluate(x_test, y_test_cat, verbose=0)
print(f"Test accuracy: {test_acc:.4f} | Test loss: {test_loss:.4f}")

y_pred = np.argmax(model.predict(x_test, verbose=0), axis=1)

In [ ]:
cm = confusion_matrix(y_test, y_pred)

plt.figure(figsize=(9, 7))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues")
plt.title("Confusion matrix (rows = true digit, columns = predicted digit)")
plt.xlabel("Predicted")
plt.ylabel("True")
plt.show()

In [ ]:
# Which digits does the model struggle with most?
errors_per_digit = cm.sum(axis=1) - np.diag(cm)
error_rate = errors_per_digit / cm.sum(axis=1)
for digit in np.argsort(error_rate)[::-1][:3]:
    print(f"Digit {digit}: {errors_per_digit[digit]} errors ({error_rate[digit]:.1%})")

# Most frequent confusions (true → predicted)
cm_off = cm.copy()
np.fill_diagonal(cm_off, 0)
flat = np.argsort(cm_off.ravel())[::-1][:5]
print("\nMost common mistakes:")
for idx in flat:
    true_d, pred_d = divmod(idx, 10)
    print(f"  {true_d} predicted as {pred_d}: {cm_off[true_d, pred_d]} times")

print()
print(classification_report(y_test, y_pred, digits=3))

In [ ]:
# Some misclassified test images
wrong = np.where(y_pred != y_test)[0][:10]
plt.figure(figsize=(12, 4))
for i, idx in enumerate(wrong):
    plt.subplot(2, 5, i + 1)
    plt.imshow(x_test[idx], cmap="gray")
    plt.title(f"True {y_test[idx]} → pred {y_pred[idx]}", color="red", fontsize=9)
    plt.axis("off")
plt.suptitle("Examples of misclassified digits")
plt.tight_layout()
plt.show()

## 5. Basic Hyperparameter Tuning Experiment
Comparing a few settings for layer sizes and learning rate (5 epochs each, to keep it quick).

In [ ]:
experiments = [
    {"hidden1": 64,  "hidden2": 32,  "learning_rate": 0.001},
    {"hidden1": 128, "hidden2": 64,  "learning_rate": 0.001},
    {"hidden1": 256, "hidden2": 128, "learning_rate": 0.001},
    {"hidden1": 128, "hidden2": 64,  "learning_rate": 0.01},
]

results = []
for params in experiments:
    m = build_model(**params)
    h = m.fit(x_train, y_train_cat, epochs=5, batch_size=64, validation_split=0.1, verbose=0)
    results.append({**params, "best_val_accuracy": max(h.history["val_accuracy"])})

import pandas as pd
pd.DataFrame(results).sort_values("best_val_accuracy", ascending=False)

## Summary
- A simple fully connected network (784 → 128 → 64 → 10) reaches roughly **97–98% test accuracy** after 10 epochs.
- The loss curves show the model learns quickly in the first epochs; after that, validation performance levels off while training keeps improving, the beginning of overfitting (more epochs wouldn't help much; dropout or early stopping would).
- The confusion matrix shows most digits are recognised very reliably. The mistakes are usually between digits that **look alike** when handwritten (typical examples: 4 ↔ 9, 3 ↔ 5, 7 ↔ 2). Run the cells above to see the exact pairs for your training run.
- In the tuning experiment, bigger hidden layers help a little, while a learning rate that is **too high (0.01)** usually makes results worse and less stable.
- Because a fully connected network ignores the 2D structure of the image, a **convolutional neural network (CNN)** would be the natural next step to go above 99%.